In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import missingno as msno
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import statsmodels.api as sm
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.svm import SVR
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from scipy.stats import ttest_rel
from sklearn.compose import TransformedTargetRegressor

In [2]:
df_lof = pd.read_csv('lof.csv')

In [3]:
X = df_lof.drop(['Dev Time (Days)', 'issue_key','Current Status'], axis=1)
y = np.log1p(df_lof['Dev Time (Days)'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

print("Model trained successfully!")
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
print(f"Mean Absolute Error (MAE): {mae:.2f}")

r2 = r2_score(y_test, predictions)
print(f"R-squared Score: {r2:.2f}")

imp = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(imp.to_string(max_rows=None))

Model trained successfully!
Mean Absolute Error (MAE): 0.46
R-squared Score: 0.33
Start Date                                  4.975504e-01
total_swag                                  1.199250e-01
impacted_area_(none)                        2.610430e-02
impacted_area_Box Test                      1.639577e-02
fix_version_(no version)                    1.477633e-02
impacted_area_Low Level                     1.279971e-02
impacted_area_SWBB Android Core / Common    1.250851e-02
technology_(none)                           1.043975e-02
technology_Aloha                            1.032456e-02
impacted_area_CPS/RM                        1.027243e-02
subtask_count                               1.022295e-02
impacted_area_Blackbox Cloud                9.646439e-03
affects_version_(no version)                9.643312e-03
technology_APX Next                         9.294106e-03
impacted_area_Ergo                          9.236955e-03
One list                                    8.518271e-03
impact

# Test Multiple Imputation Methods for total_swag (After Isolation Forest)

In [4]:
df = df_lof.copy()
num_cols = df.select_dtypes(include='number').columns
assert 'total_swag' in num_cols, "total_swag must be numeric."
df_num = df[num_cols]

In [5]:
obs_idx = df_num['total_swag'].dropna().index
rng = np.random.default_rng(42)
val_size = max(1, int(0.2 * len(obs_idx)))
val_idx = rng.choice(obs_idx, size=val_size, replace=False)
df_test = df_num.copy()
y_true = df_test.loc[val_idx, 'total_swag'].to_numpy()
df_test.loc[val_idx, 'total_swag'] = np.nan

Bayesian Ridge (features scaled)

In [6]:
est = make_pipeline(StandardScaler(), BayesianRidge())
m = 20
imputed_preds =[]

for seed in range(m):
    imp = IterativeImputer(estimator=est, sample_posterior = True, random_state = seed, max_iter = 100, skip_complete=True)
    imputed = imp.fit_transform(df_test)
    imputed_df = pd.DataFrame(imputed, columns = df_test.columns, index = df_test.index)
    imputed_preds.append(imputed_df.loc[val_idx, 'total_swag'].to_numpy())

imputed_preds = np.vstack(imputed_preds)
y_pred = imputed_preds.mean(axis=0)
mae = mean_absolute_error(y_true,y_pred)
rmse = mean_squared_error(y_true, y_pred)
print(f"MAE: {mae:.3f} RMSE: {rmse:.3f} Mean imputation SD: {imputed_preds.std(axis=0).mean():.3f}")

MAE: 7961.593 RMSE: 147514588.131 Mean imputation SD: 21357.324


Linear Regression

In [7]:
m = 20
imputed_preds =[]

for seed in range(m):
    imp = IterativeImputer(estimator=LinearRegression(), sample_posterior = False, random_state = seed, max_iter = 100, skip_complete=True)
    imputed = imp.fit_transform(df_test)
    imputed_df = pd.DataFrame(imputed, columns = df_test.columns, index = df_test.index)
    imputed_preds.append(imputed_df.loc[val_idx, 'total_swag'].to_numpy())

imputed_preds = np.vstack(imputed_preds)
y_pred = imputed_preds.mean(axis=0)
mae = mean_absolute_error(y_true,y_pred)
rmse = mean_squared_error(y_true, y_pred)
print(f"MAE: {mae:.3f} RMSE: {rmse:.3f} Mean imputation SD: {imputed_preds.std(axis=0).mean():.3f}")

MAE: 9929.159 RMSE: 192252965.739 Mean imputation SD: 0.000


Decision Tree

In [8]:
m = 20
imputed_preds =[]

for seed in range(m):
    imp = IterativeImputer(estimator=DecisionTreeRegressor(random_state=seed), sample_posterior = False, random_state = seed, max_iter = 100, skip_complete=True)
    imputed = imp.fit_transform(df_test)
    imputed_df = pd.DataFrame(imputed, columns = df_test.columns, index = df_test.index)
    imputed_preds.append(imputed_df.loc[val_idx, 'total_swag'].to_numpy())

imputed_preds = np.vstack(imputed_preds)
y_pred = imputed_preds.mean(axis=0)
mae = mean_absolute_error(y_true,y_pred)
rmse = mean_squared_error(y_true, y_pred)
print(f"MAE: {mae:.3f} RMSE: {rmse:.3f} Mean imputation SD: {imputed_preds.std(axis=0).mean():.3f}")

MAE: 4991.610 RMSE: 237620777.241 Mean imputation SD: 1218.875


Random Forest

In [9]:
m = 20
imputed_preds =[]

for seed in range(m):
    imp = IterativeImputer(estimator=RandomForestRegressor(n_estimators=200, random_state=seed, n_jobs=-1), sample_posterior = False, random_state = seed, max_iter = 100, skip_complete=True)
    imputed = imp.fit_transform(df_test)
    imputed_df = pd.DataFrame(imputed, columns = df_test.columns, index = df_test.index)
    imputed_preds.append(imputed_df.loc[val_idx, 'total_swag'].to_numpy())

imputed_preds = np.vstack(imputed_preds)
y_pred = imputed_preds.mean(axis=0)
mae = mean_absolute_error(y_true,y_pred)
rmse = mean_squared_error(y_true, y_pred)
print(f"MAE: {mae:.3f} RMSE: {rmse:.3f} Mean imputation SD: {imputed_preds.std(axis=0).mean():.3f}")

MAE: 4069.551 RMSE: 124167755.331 Mean imputation SD: 437.547


Extra Trees

In [10]:
m = 20
imputed_preds =[]

for seed in range(m):
    imp = IterativeImputer(estimator=ExtraTreesRegressor(n_estimators=200, random_state=seed, n_jobs=-1), sample_posterior = False, random_state = seed, max_iter = 100, skip_complete=True)
    imputed = imp.fit_transform(df_test)
    imputed_df = pd.DataFrame(imputed, columns = df_test.columns, index = df_test.index)
    imputed_preds.append(imputed_df.loc[val_idx, 'total_swag'].to_numpy())

imputed_preds = np.vstack(imputed_preds)
y_pred = imputed_preds.mean(axis=0)
mae = mean_absolute_error(y_true,y_pred)
rmse = mean_squared_error(y_true, y_pred)
print(f"MAE: {mae:.3f} RMSE: {rmse:.3f} Mean imputation SD: {imputed_preds.std(axis=0).mean():.3f}")

MAE: 3485.606 RMSE: 99621280.885 Mean imputation SD: 390.670


Gradient boosting

In [11]:
m = 20
imputed_preds =[]

for seed in range(m):
    imp = IterativeImputer(estimator=GradientBoostingRegressor(random_state=seed), sample_posterior = False, random_state = seed, max_iter = 100, skip_complete=True)
    imputed = imp.fit_transform(df_test)
    imputed_df = pd.DataFrame(imputed, columns = df_test.columns, index = df_test.index)
    imputed_preds.append(imputed_df.loc[val_idx, 'total_swag'].to_numpy())

imputed_preds = np.vstack(imputed_preds)
y_pred = imputed_preds.mean(axis=0)
mae = mean_absolute_error(y_true,y_pred)
rmse = mean_squared_error(y_true, y_pred)
print(f"MAE: {mae:.3f} RMSE: {rmse:.3f} Mean imputation SD: {imputed_preds.std(axis=0).mean():.3f}")

MAE: 4330.432 RMSE: 108574097.477 Mean imputation SD: 131.705


K Neighbours

In [12]:
m = 20
imputed_preds =[]

for seed in range(m):
    imp = IterativeImputer(estimator=KNeighborsRegressor(n_neighbors=5), sample_posterior = False, random_state = seed, max_iter = 100, skip_complete=True)
    imputed = imp.fit_transform(df_test)
    imputed_df = pd.DataFrame(imputed, columns = df_test.columns, index = df_test.index)
    imputed_preds.append(imputed_df.loc[val_idx, 'total_swag'].to_numpy())

imputed_preds = np.vstack(imputed_preds)
y_pred = imputed_preds.mean(axis=0)
mae = mean_absolute_error(y_true,y_pred)
rmse = mean_squared_error(y_true, y_pred)
print(f"MAE: {mae:.3f} RMSE: {rmse:.3f} Mean imputation SD: {imputed_preds.std(axis=0).mean():.3f}")

MAE: 6434.135 RMSE: 187387602.946 Mean imputation SD: 0.000


Choose Gradient Boosting

In [15]:
use_lof = df_lof.drop(['issue_key', 'Current Status'], axis=1)

imputer_lof = IterativeImputer(
    estimator = ExtraTreesRegressor(n_estimators=200, random_state=seed, n_jobs=-1),
    max_iter = 10,
    random_state = 42
)

df_lof_imputed_values = imputer_lof.fit_transform(use_lof)
df_lof_final = pd.DataFrame(df_lof_imputed_values, columns=use_lof.columns, index=use_lof.index)

In [16]:
df_lof_final.to_csv('imputed_lof.csv', index=False)